# Big heading
## Smaller heading
### Even smaller

- item one
- item two
  - nested item (indent with 2 spaces)

1. first
2. second
3. third

**bold**
*italic*
`inline code`

$z = \frac{x - \mu}{\sigma}$

```python
x = 1 + 1
```

# Step 1: Retrieving data

## Pull data from yfinance. 

### In **yf.download**, we accept the parameters: 
  1. tickers= --> accepts a single string or a list of strings
  2. start= --> string start date inclusive in the format YYYY-MM-DD
  3. end= --> string end date inclusive in the formaat YYYY-MM-DD
  4. interval= --> bar size, i.e. each row has what length of time worth of data. If interval="1h", each row has one hour of price data. We generally work with 1d
  5. auto_adjust=True(default) --> for eg, if a company does a 2-for-1 stock split, it means every existing share doubles into 2 new ones. Stock price would be halved. If we have auto_adjust=False, it would look like the stock had a 50% crash. Autoadjust accounts for these fluctuations

  - yfinance also imports pandas internally, so the returned object from .download is actually a pandas dataframe, and we can use pandas methods on it

## Why use Close data?
We COULD use the other data (except Volume because it is not price related), but Close is standard because:
1. Represents the final agreed price of the day after all trading activity
2. It establishes the true benchmark for a stock's value until the next day

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

In [ ]:
tickers = ["KO", "PEP"]
START, END = "2015-01-01", "2024-01-01"
df = yf.download(tickers, start=START, end=END) #returns a pandas dataframe object 
prices = df["Close"]


[*********************100%***********************]  2 of 2 completed


# Step 2: Data Cleaning

## A) Filling up NaN values with .ffill() and .dropna()
### .ffill()
- uses data from the previous day to fill up a NaN value. 
- If a stock didn't trade today, the last known price is the best estimate of the value, unless that last known price is from too long ago
- we can pass the parameter limit=. This sets a limit on the number of days to look back to since ffill by default looks back as far as possible to get the previous data
### .dropna()
- removes any NaN values


- here, we use .ffill() AND THEN .dropna(), but why not .dropna() first before .ffill()?
  1. By filling before dropping NaN values, we prevent rows where KO has a valid value but PEP does not from being removed, meaning less data is removed
  2. Initially, I wondered wouldn't it be better to NaN values dropped first, before we ffill since we can use a price from multiple days ago if the previous day's price is also NaN. However, this is unideal as a 2-day gap is generally suspicious enough that it is safer to remove the price than fabricate it


## B) Minimum History Requirement
- here for KO and PEP, we are running a ***cointegration statistical test***. 

### What is a Cointegration Statistical Test
- It is essentially testing "is the relationship between KO and PEP real or could it be random?"
- We need sufficient data to ensure the p-value our statistical test produces has enough statistical power, if not the the p-value would be inconclusive and it would be unwise to act on it since any relationship shown could be from chance without sufficient data points to prove it is not. 
- Generally there are 2 types of Cointegration tests, the Engle-Granger Test and the Johansen Test
  - To put it simply, the Engle-Granger Test is used for 2 variables, while the Johansen Test is used for analysis of multiple time series at the same time, i.e. more than two variables

### Why check for 2 years worth of trading data specifically?
- To cover a full market cycle with a reasonable chance to capture both bullish and bearish market conditions. A relationship that only holds in one regime is not useful
- We will be using the Engle-Granger Test which generally needs a few hundered observations to produce reliable p-values
- The relationship needs time to express itself. Pairs trading relies on the spread mean-reverting (returning to their long-term historical means over time). If I only have say 3 months of data, I might not have seen enough full cycles of the spread diverging and reverting to know its a real pattern
- Note the 2 years minimum is just the floor below which statistics become unreliable, we generally aim for at least 5 years worth of data

### In the code, why use the assert function?
- The program will terminate immediately when our quantity of data is insufficient, allowing us to adjust


## C) Synchronisation Test
- Here, we are checking that for each row, the trading dates for KO and PEP are the same
- This is important as further down, we are running code similar to what we have below:
```python
spread = prices["PEP"] - hedge_ratio * prices["KO"]
```
  - Pandas aligns by index automatically before doing the arithmetic. So if KO has a date that PEP doesn't, pandas would return NaN for that row rather than using the wrong price.
  - However, this means that misaligned dates would silently introduce NaNs into the spread, which would lead to issues downstream
- Generally, for large stocks in the same exchange, it is expected for the stocks to have the same date indices. However, this issue can be more prevalent in stocks traded in different exchanges, or for stocks which are smaller

### .index in a time series dataframe
- Usually, indices in a pandas dataframe are integers from 0 onwards, but when we load time series data like what we have here, the indices become dates in the form of YYYY-MM-DD, so we can naturally confirm the dates are the same by ensuring the indices are the same

### prices[ticker].index
- We are returning a Series (single column) of data from the ticker of our choice. In this case, the column we are returning is the column with the dates(indices)

In [ ]:
# forward filling and dropping NaN values
prices = prices.ffill()
prices = prices.dropna()

In [ ]:
# Minimum History Requirement check (I am doing a check for 5 years here to make my data more robust)
MIN_YEARS = 5
assert len(prices) >= 252 * MIN_YEARS, f"Need at least {MIN_YEARS} years of data"

#will produce 
# AssertionError: Need at least __ years of data
# if not enough data

In [ ]:
#Synchronisation check
#Note .equals is a pandas function, but yfinance imports it internally and the Dataframe it returns is a panddas object
assert prices["KO"].index.equals(prices["PEP"].index), "Tickers have misaligned trading dates"

In [21]:
# Outlier check
returns = prices.pct_change()
mean = returns.mean()
std = returns.std()
mask = (returns - mean).abs() > 5 * std
print(mask.sum())

Ticker
KO     12
PEP     8
dtype: int64
